# Pipe and Filter — Exemplo Executável

**Autores:** Thiago Leal, Emerson Silva, José Victor, Randson Bredley, Thiago Patriota e Gustavo Henrique (Grupo 3)

Este notebook demonstra o estilo arquitetural **Pipe and Filter** por meio de um pipeline de processamento de texto. Primeiro, implementamos filtros independentes em lote; em seguida, encadeamos geradores para processar as entradas sob demanda. Por fim, mostramos uma violação de independência entre filtros.

**Referências:**
- Shaw & Garlan, *Software Architecture: Perspectives on an Emerging Discipline*, 1996, cap. 2
- Bass, Clements & Kazman, *Software Architecture in Practice*, 4ª ed., 2021, cap. 13
- Buschmann et al., *Pattern-Oriented Software Architecture*, Vol. 1, 1996, cap. 2

## 1. Dependências e dados de entrada

O exemplo usa apenas a biblioteca padrão do Python. O texto abaixo é pequeno para facilitar a leitura dos resultados.

In [ ]:
import re
from collections import Counter
from collections.abc import Callable, Iterator
from typing import Any

texto_bruto = """
A arquitetura de software define a estrutura fundamental de um sistema.
Ela determina como os componentes se organizam, como se comunicam e quais
restrições governam o projeto. Um estilo arquitetural como o Pipe and Filter
decompõe o processamento em etapas independentes, cada uma responsável por
uma transformação específica. Os filtros não conhecem uns aos outros;
eles apenas recebem dados, transformam e produzem resultados. Essa
independência permite que filtros sejam adicionados, removidos ou
substituídos sem afetar o restante do pipeline. O estilo é amplamente
utilizado em compiladores, pipelines de ETL, processamento de sinais
e sistemas de análise de logs.
"""

print("Dados de entrada carregados:", len(texto_bruto.split()), "palavras")

## 2. Implementação correta — filtros independentes

Cada filtro transforma sua entrada e devolve um resultado, sem acessar o estado interno dos outros filtros. O pipeline conecta os filtros usando a saída de cada etapa como entrada da seguinte.

In [ ]:
def normalizar(texto: str) -> str:
    """Converte para minúsculas e remove pontuação."""
    return re.sub(r"[^\w\s]", "", texto.lower())


def tokenizar(texto: str) -> list[str]:
    """Divide o texto em palavras individuais."""
    return texto.split()


STOP_WORDS_PT = {
    "a", "o", "e", "é", "de", "do", "da", "dos", "das", "em", "um",
    "uma", "os", "as", "no", "na", "se", "que", "como", "por", "para", "com",
}


def filtrar_stop_words(palavras: list[str]) -> list[str]:
    """Remove palavras comuns sem valor semântico para este exemplo."""
    return [palavra for palavra in palavras if palavra not in STOP_WORDS_PT and len(palavra) > 1]


def contar_frequencias(palavras: list[str]) -> dict[str, int]:
    """Conta a frequência de cada palavra."""
    return dict(Counter(palavras).most_common())


def pipeline(dado: Any, *filtros: Callable[..., Any]) -> Any:
    """Executa os filtros em sequência, conectando suas entradas e saídas."""
    resultado = dado
    for filtro in filtros:
        resultado = filtro(resultado)
    return resultado

print("Filtros e pipeline definidos.")

In [ ]:
resultado_lote = pipeline(
    texto_bruto,
    normalizar,
    tokenizar,
    filtrar_stop_words,
    contar_frequencias,
)

print("Cinco palavras mais frequentes (lote):")
for palavra, frequencia in list(resultado_lote.items())[:5]:
    print(f"  {palavra:.<20} {frequencia}")

## 3. Processamento sob demanda com geradores

No processamento em lote, cada etapa cria uma string, lista ou dicionário com os resultados intermediários. Geradores permitem que normalização, tokenização e filtragem produzam itens sob demanda, sem materializar a lista completa de tokens.

A agregação final com `Counter`, porém, mantém uma entrada por palavra distinta. Portanto, seu uso de memória cresce com o vocabulário (**O(U)**, onde *U* é o número de palavras distintas); não é correto afirmar que uma contagem exata sempre usa memória constante. Para uma entrada real grande, a fonte também deve ser lida incrementalmente, por exemplo iterando sobre um arquivo, em vez de carregar todo o arquivo numa string.

In [ ]:
def normalizar_stream(linhas: Iterator[str]) -> Iterator[str]:
    """Produz linhas normalizadas uma a uma."""
    for linha in linhas:
        linha_limpa = re.sub(r"[^\w\s]", "", linha.lower())
        if linha_limpa.strip():
            yield linha_limpa


def tokenizar_stream(linhas: Iterator[str]) -> Iterator[str]:
    """Produz palavras sob demanda a partir das linhas."""
    for linha in linhas:
        for match in re.finditer(r"\w+", linha):
            yield match.group()


def filtrar_stop_words_stream(palavras: Iterator[str]) -> Iterator[str]:
    """Filtra palavras enquanto o fluxo é consumido."""
    for palavra in palavras:
        if palavra not in STOP_WORDS_PT and len(palavra) > 1:
            yield palavra


def contar_frequencias_stream(palavras: Iterator[str]) -> dict[str, int]:
    """Consome o fluxo e mantém a contagem de cada palavra distinta."""
    return dict(Counter(palavras).most_common())


# A amostra já está em memória; iteramos suas linhas para demonstrar o encadeamento.
# Para arquivos grandes, use (linha for linha in arquivo_aberto) como fonte.
fonte_linhas = iter(texto_bruto.splitlines())
stream_normalizado = normalizar_stream(fonte_linhas)
stream_tokenizado = tokenizar_stream(stream_normalizado)
stream_filtrado = filtrar_stop_words_stream(stream_tokenizado)
resultado_stream = contar_frequencias_stream(stream_filtrado)

print("Cinco palavras mais frequentes (streaming):")
for palavra, frequencia in list(resultado_stream.items())[:5]:
    print(f"  {palavra:.<20} {frequencia}")

assert resultado_stream == resultado_lote
print()
print("Contagens em lote e streaming são iguais.")

## 4. Violação de independência — acoplamento entre filtros

A independência é violada quando um filtro mantém uma referência a outro filtro e consulta diretamente seu estado interno. Isso cria uma dependência que dificulta substituir e testar os componentes separadamente.

In [ ]:
class NormalizadorAcoplado:
    def __init__(self, idioma: str = "pt") -> None:
        self.idioma = idioma

    def __call__(self, texto: str) -> str:
        return normalizar(texto)


class ContadorAcoplado:
    def __init__(self, normalizador: NormalizadorAcoplado) -> None:
        self.normalizador = normalizador

    def __call__(self, palavras: list[str]) -> dict[str, int]:
        # Esta consulta cria dependência direta do estado de outro filtro.
        idioma = self.normalizador.idioma
        stop_words = STOP_WORDS_PT if idioma == "pt" else set()
        filtradas = [
            palavra for palavra in palavras
            if palavra not in stop_words and len(palavra) > 1
        ]
        return dict(Counter(filtradas).most_common(5))


normalizador_acoplado = NormalizadorAcoplado(idioma="pt")
contador_acoplado = ContadorAcoplado(normalizador_acoplado)
resultado_acoplado = pipeline(
    texto_bruto,
    normalizador_acoplado,
    tokenizar,
    contador_acoplado,
)

print("Cinco palavras mais frequentes (exemplo acoplado):")
for palavra, frequencia in resultado_acoplado.items():
    print(f"  {palavra:.<20} {frequencia}")

assert resultado_acoplado == dict(Counter(resultado_lote).most_common(5))